<div>
<div style="font-size: 0.85em; letter-spacing: 0.08em; text-transform: uppercase; color: #1c4ed8; font-weight: 700; margin-bottom: 0.5em;">ML Research &middot; Rate Plan Design</div>
<h1 style="margin: 0 0 0.3em; font-weight: 800; letter-spacing: -0.01em;">From plan choice models to a launch decision</h1>
<p style="margin-top: 0; font-size: 1.2em; color: #444;">Forecasting where a lower-priced wireless plan would draw its customers from, and which price, features, and eligibility rules keep it from cannibalizing higher-margin plans</p>
</div>

<div style="border: 2px solid #a94442; padding: 20px 30px;">
<h2>Disclosure</h2>
<ul style="margin-left: 0; padding-left: 1.25em;">
<li>All data and modeled results are synthetic. The approximately 80,000-household market, its plan history, the conjoint survey, and the counterfactual launch were generated and validated in <code>data/00_generate_and_validate_data.ipynb</code>.</li>
<li>Plan names, prices, and features are illustrative and do not describe any carrier's offers.</li>
<li>The modeling notebooks read only tables a carrier could actually hold. The withheld answer key (true taste classes, part-worths, and counterfactual launch outcomes) is loaded only to score forecasts, never to fit or tune a model.</li>
<li>The study is written as two notebooks: <code>01_choice_model_estimation.ipynb</code> (sections 01–04) and <code>02_launch_forecast.ipynb</code> (sections 05–08).</li>
</ul>
</div>

<hr>
<div>
<h2>Abstract</h2>
<ul style="margin-left: 0; padding-left: 1.25em;">
<li><em>To be written once results are final.</em></li>
</ul>
</div>

<hr>
<div>
<h2>Introduction</h2>
<p><strong>Business Question:</strong> A carrier is considering a lower-priced plan, <code>essentials</code>, below its existing <code>premium</code>, <code>standard</code>, and <code>value</code> plans. How many customers would take it, where would they come from, and which price, feature bundle, and eligibility rules make it add margin rather than cannibalize higher-margin plans?</p>
<p><strong>Data and Scope:</strong> The study uses 24 months of pre-launch history for about 51,000 customer accounts, public competitor pricing, and a 4,000-respondent conjoint survey that includes non-customers. It evaluates 12-month contribution margin across 108 launch configurations (9 prices × 4 feature bundles × 3 eligibility rules), including customers won from competitors and churners saved by the cheaper plan.</p>
<p><strong>Study Design:</strong> The study runs across two notebooks. The first estimates how customers value price and plan features; the second turns those estimates into a launch forecast, scores it against the withheld answer key, and makes the recommendation.</p>
<table style="width: 100%; border-collapse: collapse;">
<thead>
<tr>
<th style="text-align: left;">Section</th>
<th style="text-align: left;">Notebook</th>
<th style="text-align: left;">Business Question Answered</th>
</tr>
</thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;">01 · The Revealed-Preference Baseline</td><td style="text-align: left; vertical-align: top;">Estimation</td><td style="text-align: left; vertical-align: top;">What can 24 months of plan history alone say about price and feature sensitivity, and where does it run out?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">02 · The Stated-Preference Mixed Logit</td><td style="text-align: left; vertical-align: top;">Estimation</td><td style="text-align: left; vertical-align: top;">How much do customers value each plan feature relative to price, and how much does that vary between customers?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">03 · The Joint RP–SP Nested Model</td><td style="text-align: left; vertical-align: top;">Estimation</td><td style="text-align: left; vertical-align: top;">When stated and revealed choices are combined and survey bias is corrected, which plans substitute for which?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">04 · Latent Taste Classes</td><td style="text-align: left; vertical-align: top;">Estimation</td><td style="text-align: left; vertical-align: top;">Which customer groups value which features, and can every account be assigned to one from carrier data?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">05 · Back-Test on the Value Launch</td><td style="text-align: left; vertical-align: top;">Forecast</td><td style="text-align: left; vertical-align: top;">Fit only on the months before <code>value</code> launched, would the approach have forecast where its customers came from?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">06 · The Launch Simulation</td><td style="text-align: left; vertical-align: top;">Forecast</td><td style="text-align: left; vertical-align: top;">For each launch configuration, how many customers take <code>essentials</code>, from which plans and carriers, and what is the 12-month margin effect?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">07 · Scoring Against the Answer Key</td><td style="text-align: left; vertical-align: top;">Forecast</td><td style="text-align: left; vertical-align: top;">How far off is each model's forecast from the true launch outcome, and does a flat logit really overstate cannibalization?</td></tr>
<tr><td style="text-align: left; vertical-align: top;">08 · Recommendation and Rollout Design</td><td style="text-align: left; vertical-align: top;">Forecast</td><td style="text-align: left; vertical-align: top;">Which price, bundle, and eligibility rule should launch, and how should the rollout be designed to measure the result?</td></tr>
</tbody>
</table>
<p><strong>Model Requirements:</strong> A cheaper plan's financial case depends on its source of volume, not its take rate, so the models must represent realistic substitution between plans rather than the proportional substitution a flat multinomial logit imposes. Every forecast is scored against the withheld answer key before it informs the recommendation.</p>
</div>

<hr>
<div>
<h2>Initialize Estimation Environment</h2>
<p>Initializes the shared imports, plotting style, file paths, constants, and helper functions used throughout sections 01–04. Model-facing tables load through <code>load()</code>, which refuses answer-key tables; each section then loads only the data it requires.</p>
</div>

In [1]:
# Standard Library
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import logsumexp, softmax
from scipy.stats import norm, qmc
from IPython.display import display

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 40)

In [2]:
# Plotting Theme and Color Palette
SURFACE = '#ffffff'
INK = '#0b0b0b'
MUTED = '#898781'
GRID = '#e1e0d9'
BASELINE = '#c3c2b7'
ACCENT = '#2a78d6'
ACCENT2 = '#eb6834'
RED = '#e34948'
TEAL = '#1f9e89'
PURPLE = '#7b4fc9'

plt.rcParams.update({
    'figure.dpi': 110,
    'figure.facecolor': SURFACE,
    'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE,
    'axes.edgecolor': BASELINE,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'grid.color': GRID,
    'grid.linewidth': 0.8,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'font.size': 9,
    'text.color': INK,
    'axes.labelcolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
})

In [3]:
# Data Loading and Export Helpers
DATA = Path('data/synthetic')
OUTPUTS = Path('outputs')
OUTPUTS.mkdir(exist_ok = True)

MODEL_TABLES = [
    'accounts', 'usage_panel', 'plan_events', 'plan_catalog', 'price_history', 'markets',
    'conjoint_responses', 'conjoint_respondents',
]
ANSWER_KEY_TABLES = [
    'ground_truth', 'true_plan_events', 'value_launch_counterfactual',
    'counterfactual_launch', 'counterfactual_grid', 'generator_parameters',
]


def load(name):
    """Loads a table the carrier could actually hold. Answer-key tables are refused."""

    if name in ANSWER_KEY_TABLES:
        raise PermissionError(f'{name} is part of the withheld answer key; use load_answer_key() only when scoring')
    if name not in MODEL_TABLES:
        raise KeyError(f'unknown table {name!r}')

    return pd.read_parquet(DATA / f'{name}.parquet')


def load_answer_key(name):
    """Loads a withheld table. Used only in scoring cells, never to fit or tune a model."""

    if name not in ANSWER_KEY_TABLES:
        raise KeyError(f'{name!r} is not an answer-key table')

    return pd.read_parquet(DATA / f'{name}.parquet')


def save_output(df, name):

    path = OUTPUTS / f'{name}.parquet'
    df.to_parquet(path, index = False)

    return path

In [4]:
# Domain Constants
ALTS = ['premium', 'standard', 'value', 'essentials', 'comp_premium', 'comp_value', 'outside']
OWN_PLANS = ['premium', 'standard', 'value', 'essentials']
ORIGINS = ['premium', 'standard', 'value', 'competitor', 'outside']
ORIGIN_OF = {
    'premium': 'premium', 'standard': 'standard', 'value': 'value',
    'comp_premium': 'competitor', 'comp_value': 'competitor', 'outside': 'outside',
}

N_MONTHS = 24
N_HORIZON = 12
ADDL_LINE_RATE = 0.65

ESSENTIALS_PRICES = np.arange(35.0, 55.1, 2.5)
ESSENTIALS_BUNDLES = ['lean', 'lean_hotspot', 'lean_30gb', 'lean_hd']
ELIGIBILITY = ['open', 'no_multiline_discount', 'new_lines_only']
CONFIG_KEYS = ['bundle', 'eligibility', 'essentials_price']
REFERENCE_CONFIG = ('lean', 'open', 45.0)

<hr style="border: none; border-top: 2px solid #4A4A4A;">
<div>
<h1>01 · The Revealed-Preference Baseline</h1>
<p><strong>Purpose:</strong> Estimates plan choice from 24 months of carrier history using only observable data. It establishes what revealed choices alone can identify, what they cannot, and the flat-logit substitution pattern that later sections are measured against.</p>
<p><strong>Architecture:</strong></p>
<table style="width: 100%; border-collapse: collapse;">
<thead><tr><th style="text-align: left;">Stage</th><th style="text-align: left;">Method</th><th style="text-align: left;">Reasoning</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;">1</td><td style="text-align: left; vertical-align: top;">Choice occasions from observable triggers</td><td style="text-align: left; vertical-align: top;">Whether a customer reconsidered their plan is never recorded; device upgrades, throttling, care contacts, and promotions stand in for it.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">2</td><td style="text-align: left; vertical-align: top;">Censored regression on usage above the cap</td><td style="text-align: left; vertical-align: top;">Observed usage understates demand wherever a plan throttles, so feature value built on raw usage would be biased toward the current plan.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">3</td><td style="text-align: left; vertical-align: top;">Conditional logit with plan × line-count constants</td><td style="text-align: left; vertical-align: top;">A transparent baseline whose price coefficient rests on the designed price events rather than on differences between household types.</td></tr>
</tbody>
</table>
</div>

In [5]:
# Load Estimation Inputs
accounts = load('accounts')
usage_panel = load('usage_panel')
plan_events = load('plan_events')
plan_catalog = load('plan_catalog')
price_history = load('price_history')
markets = load('markets')

print(
    f'{len(accounts):,} accounts · {len(usage_panel):,} account-months · '
    f'{len(plan_events):,} plan events · {usage_panel.month.nunique()} months'
)

50,927 accounts · 1,017,839 account-months · 30,646 plan events · 24 months


<hr>
<div>
<h2>1 · Choice Occasions from Observable Triggers</h2>
<p><strong>Purpose:</strong> Defines the account-months treated as choice occasions, since consideration is unobserved.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>2 · Recovering Demand above the Cap</h2>
<p><strong>Purpose:</strong> Estimates each account's underlying data demand from usage that is censored by throttling.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>3 · Feature Surplus from Estimated Demand</h2>
<p><strong>Purpose:</strong> Converts plan attributes into account-specific costs: expected throttled GB, hotspot shortfall, and uncovered roaming days under each plan.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>4 · Flat Conditional Logit</h2>
<p><strong>Purpose:</strong> Fits the baseline choice model and reports how precisely each coefficient is identified.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>5 · What History Cannot Identify</h2>
<p><strong>Purpose:</strong> Documents the limits of revealed choices: plan-level features that are collinear with plan constants, and the thin price variation behind the price coefficient.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>6 · Model Outputs and Downstream Handoff</h2>
<p><strong>Purpose:</strong> Saves choice occasions, estimated demand, and baseline coefficients for sections 03 and 05.</p>
<p><em>To be built.</em></p>
</div>

<hr style="border: none; border-top: 2px solid #4A4A4A;">
<div>
<h1>02 · The Stated-Preference Mixed Logit</h1>
<p><strong>Purpose:</strong> Estimates how much respondents value each plan feature relative to price, and how much those valuations vary between people, from the conjoint survey. This is the only source of variation in features independent of price, and the only way to value a plan nobody has seen.</p>
<p><strong>Architecture:</strong></p>
<table style="width: 100%; border-collapse: collapse;">
<thead><tr><th style="text-align: left;">Stage</th><th style="text-align: left;">Method</th><th style="text-align: left;">Reasoning</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;">1</td><td style="text-align: left; vertical-align: top;">Response-propensity weights</td><td style="text-align: left; vertical-align: top;">Some customer types answer surveys more often; unweighted estimates would describe the respondents rather than the base.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">2</td><td style="text-align: left; vertical-align: top;">Panel mixed logit by simulated maximum likelihood (scipy L-BFGS-B, analytic gradients, scrambled Halton draws)</td><td style="text-align: left; vertical-align: top;">Coefficients vary across respondents but stay fixed across one respondent's tasks. A lognormal price coefficient keeps price sensitivity negative for everyone.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">3</td><td style="text-align: left; vertical-align: top;">Holdout-task validation</td><td style="text-align: left; vertical-align: top;">Two fixed tasks excluded from estimation test whether the model predicts choices it has not seen.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">4</td><td style="text-align: left; vertical-align: top;">Individual-level conditional part-worths</td><td style="text-align: left; vertical-align: top;">Each respondent's own choices sharpen the population distribution into a person-level estimate, used later to link tastes to observable data.</td></tr>
</tbody>
</table>
</div>

In [6]:
# Load Conjoint
conjoint = load('conjoint_responses')
conjoint_respondents = load('conjoint_respondents')

print(
    f'{conjoint.respondent_id.nunique():,} respondents · '
    f'{conjoint.loc[~conjoint.is_holdout, "task"].nunique()} estimation tasks + '
    f'{conjoint.loc[conjoint.is_holdout, "task"].nunique()} holdout tasks each · '
    f'{conjoint_respondents.is_customer.mean():.0%} current customers'
)

4,000 respondents · 10 estimation tasks + 2 holdout tasks each · 75% current customers


<hr>
<div>
<h2>1 · Sample Reweighting for Response Bias</h2>
<p><strong>Purpose:</strong> Weights respondents so the sample matches the customer base and the competitor market on observable fields.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>2 · Model Specification and Halton Draws</h2>
<p><strong>Purpose:</strong> Sets the utility specification, coefficient distributions, and simulation draws.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>3 · Simulated Maximum Likelihood</h2>
<p><strong>Purpose:</strong> Fits the mixed logit and checks that the estimates are stable as the number of draws grows.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>4 · Holdout Validation</h2>
<p><strong>Purpose:</strong> Compares predicted and observed choice shares on the two holdout tasks, for customers and non-customers.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>5 · Individual-Level Part-Worths</h2>
<p><strong>Purpose:</strong> Computes each respondent's conditional part-worths and willingness to pay.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>6 · Model Outputs and Downstream Handoff</h2>
<p><strong>Purpose:</strong> Saves population and individual-level estimates for sections 03 and 04.</p>
<p><em>To be built.</em></p>
</div>

<hr style="border: none; border-top: 2px solid #4A4A4A;">
<div>
<h1>03 · The Joint RP–SP Nested Model</h1>
<p><strong>Purpose:</strong> Combines revealed and stated choices in one model, so the survey supplies feature trade-offs, the history anchors real-world price sensitivity and plan constants, and a nested structure captures which plans substitute for which.</p>
<p><strong>Architecture:</strong></p>
<table style="width: 100%; border-collapse: collapse;">
<thead><tr><th style="text-align: left;">Stage</th><th style="text-align: left;">Method</th><th style="text-align: left;">Reasoning</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;">1</td><td style="text-align: left; vertical-align: top;">Stacked RP and SP likelihood with a relative scale parameter</td><td style="text-align: left; vertical-align: top;">Stated choices are noisier than real ones; the scale parameter lets both sources share coefficients without forcing equal noise.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">2</td><td style="text-align: left; vertical-align: top;">SP-specific price attenuation</td><td style="text-align: left; vertical-align: top;">Respondents understate price sensitivity; a separate SP price multiplier keeps that bias out of the forecast.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">3</td><td style="text-align: left; vertical-align: top;">Nested logit, with price-tier and carrier trees compared</td><td style="text-align: left; vertical-align: top;">The substitution structure is estimated and tested rather than assumed.</td></tr>
</tbody>
</table>
</div>

<hr>
<div>
<h2>1 · Stacking Revealed and Stated Choices</h2>
<p><strong>Purpose:</strong> Puts both data sources on one alternative set and one feature definition.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>2 · Scale and Price Attenuation</h2>
<p><strong>Purpose:</strong> Estimates the relative noise of stated choices and the survey's understatement of price sensitivity.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>3 · Nest Structure: Price Tier vs Carrier</h2>
<p><strong>Purpose:</strong> Fits competing nesting trees and compares them on fit and on held-out switching patterns.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>4 · Substitution Diagnostics</h2>
<p><strong>Purpose:</strong> Shows where a new plan would draw from under the nested model versus a flat logit with the same coefficients.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>5 · Model Outputs and Downstream Handoff</h2>
<p><strong>Purpose:</strong> Saves the joint model's coefficients and nesting parameters for sections 05 and 06.</p>
<p><em>To be built.</em></p>
</div>

<hr style="border: none; border-top: 2px solid #4A4A4A;">
<div>
<h1>04 · Latent Taste Classes</h1>
<p><strong>Purpose:</strong> Finds groups of customers who value features in distinctly different ways, and assigns every account to one using only carrier data, so the launch forecast and the recommendation can speak about segments rather than an average customer.</p>
<p><strong>Architecture:</strong></p>
<table style="width: 100%; border-collapse: collapse;">
<thead><tr><th style="text-align: left;">Stage</th><th style="text-align: left;">Method</th><th style="text-align: left;">Reasoning</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;">1</td><td style="text-align: left; vertical-align: top;">Latent class logit by expectation–maximization</td><td style="text-align: left; vertical-align: top;">Discrete classes with their own part-worths are easier to act on than a continuous distribution, and class count can be chosen on information criteria.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">2</td><td style="text-align: left; vertical-align: top;">Class membership as a function of observables</td><td style="text-align: left; vertical-align: top;">Membership must be predictable from lines, device, usage, and roaming for the classes to be usable on the full base.</td></tr>
<tr><td style="text-align: left; vertical-align: top;">3</td><td style="text-align: left; vertical-align: top;">Scoring the customer base</td><td style="text-align: left; vertical-align: top;">Each account gets class probabilities, which the simulation uses in place of a single average taste.</td></tr>
</tbody>
</table>
</div>

<hr>
<div>
<h2>1 · Choosing the Number of Classes</h2>
<p><strong>Purpose:</strong> Fits two to six classes and selects on BIC, holdout fit, and interpretability.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>2 · Class Part-Worths and Willingness to Pay</h2>
<p><strong>Purpose:</strong> Profiles each class by what it will pay for data, hotspot, streaming quality, roaming, and perks.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>3 · Class Membership from Observables</h2>
<p><strong>Purpose:</strong> Links class probabilities to carrier data and measures how well membership can be predicted.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>4 · Scoring the Customer Base</h2>
<p><strong>Purpose:</strong> Assigns class probabilities to every account for the launch simulation.</p>
<p><em>To be built.</em></p>
</div>

<hr>
<div>
<h2>5 · Model Outputs and Downstream Handoff</h2>
<p><strong>Purpose:</strong> Saves class part-worths, membership model, and scored accounts for notebook 02.</p>
<p><em>To be built.</em></p>
</div>